# 🔬 UltimateDocResearcher — End-to-End Demo

This notebook walks through the complete pipeline:
1. **Collect** — PDFs + web + GitHub → `data/all_docs.txt`
2. **Analyze** — quality filter + chunking → `data/all_docs_cleaned.txt`
3. **Prepare** — Q&A generation → `data/train.jsonl` + `data/val.jsonl`
4. **Train** — LoRA fine-tune on Kaggle GPU
5. **Results** — view `results/results.tsv`

> ⚡ Designed to run on **Kaggle T4 GPU** (free tier) — no local GPU needed.

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
import os, sys, subprocess

# Clone repo if running on Kaggle/Colab
if not os.path.exists('ultimate-doc-researcher'):
    subprocess.run(['git', 'clone',
        'https://github.com/yourusername/ultimate-doc-researcher.git'],
        check=True)
    os.chdir('ultimate-doc-researcher')
else:
    os.chdir('ultimate-doc-researcher')

sys.path.insert(0, '.')
print('✅ Working dir:', os.getcwd())

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
subprocess.run([
    'pip', 'install', '-q',
    'pymupdf', 'aiohttp', 'beautifulsoup4',
    'google-api-python-client', 'google-auth',
], check=True)

# GPU training deps (skip on CPU)
try:
    import torch
    if torch.cuda.is_available():
        subprocess.run(['pip', 'install', '-q', 'unsloth', 'trl', 'peft'], check=True)
        print('✅ GPU training deps installed')
except ImportError:
    pass

print('✅ Dependencies ready')

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
TOPIC = 'Claude skills optimization'
N_ITERATIONS = 3   # increase to 20+ for a full run
USE_LLM_QA = False  # set True if OPENAI_API_KEY is set

# Optionally set API keys
# os.environ['OPENAI_API_KEY'] = 'sk-...'
# os.environ['GOOGLE_API_KEY'] = '...'
# os.environ['GOOGLE_CX'] = '...'
# os.environ['GITHUB_TOKEN'] = '...'
print(f'Topic: {TOPIC}\nIterations: {N_ITERATIONS}')

## Step 1: Collect Documents

In [ ]:
from collector.ultimate_collector import UltimateCollector

collector = UltimateCollector(
    google_queries=[
        f'{TOPIC} research paper',
        f'{TOPIC} best practices 2024',
        f'anthropic claude prompt engineering',
    ],
    reddit_subreddits=['MachineLearning', 'LocalLLaMA', 'ClaudeAI'],
    github_repos=['karpathy/autoresearch', 'anthropics/anthropic-sdk-python'],
    output_dir='data/',
    min_chars=300,
)

docs = collector.run()
print(f'\n📄 Collected {len(docs)} documents')
print('Sources:', {d.source for d in docs})

## Step 2: Analyze & Clean

In [ ]:
from collector.analyzer import analyze_corpus

report = analyze_corpus('data/all_docs.txt', 'data/')
print(f"\nDocs passing quality filter: {report['docs_passing_filter']} / {report['total_docs']}")
print(f"Total chunks: {report['total_chunks']}")
print(f"Avg quality score: {report['avg_quality_score']}")
print(f"Source breakdown: {report['source_breakdown']}")

## Step 3: Prepare Q&A Training Data

In [ ]:
from autoresearch.prepare import prepare

# Generate program.md for this topic
from templates.program_templates import get_program
prog = get_program('claude-skills-optimizer')
prog.save('templates/program.md')

train_path, val_path = prepare(
    corpus_path='data/all_docs_cleaned.txt',
    output_dir='data/',
    program_md='templates/program.md',
    max_pairs=200,     # increase for better coverage
    use_llm=USE_LLM_QA,
    val_frac=0.1,
)
print(f'\n✅ train: {train_path}  val: {val_path}')

# Preview a sample
import json
with open(train_path) as f:
    sample = json.loads(f.readline())
print('\nSample Q&A:')
for m in sample['messages']:
    print(f"  [{m['role']}] {m['content'][:120]}")

## Step 4: Train (LoRA fine-tune)

> Requires GPU. On Kaggle: Runtime → GPU T4 x2

In [ ]:
import torch
if not torch.cuda.is_available():
    print('⚠️  No GPU detected — skipping training. Enable GPU on Kaggle.')
else:
    from autoresearch.train import TrainConfig, train
    
    all_metrics = []
    for i in range(N_ITERATIONS):
        print(f'\n{'='*50}')
        print(f'  Iteration {i+1}/{N_ITERATIONS}')
        print('='*50)
        
        cfg = TrainConfig(
            model_name='unsloth/Llama-3.2-3B-Instruct-bnb-4bit',
            num_train_epochs=1,
            iteration=i+1,
            topic=TOPIC,
            output_dir='/kaggle/working/models/lora',
            results_tsv='results/results.tsv',
        )
        metrics = train(cfg)
        all_metrics.append(metrics)
        print(f"  val_score = {metrics.get('val_score')}")
    
    print('\n✅ Training loop complete')

## Step 5: Results

In [ ]:
import pandas as pd
from pathlib import Path

tsv = Path('results/results.tsv')
if tsv.exists():
    df = pd.read_csv(tsv, sep='\t')
    print(df[['iteration', 'val_loss', 'val_score', 'elapsed_seconds', 'topic']].to_string())
    
    # Plot if matplotlib available
    try:
        import matplotlib.pyplot as plt
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        axes[0].plot(df['iteration'], df['val_loss'], 'o-', color='#e74c3c')
        axes[0].set_title('Val Loss over Iterations')
        axes[0].set_xlabel('Iteration')
        axes[0].set_ylabel('Val Loss')
        axes[1].plot(df['iteration'], df['val_score'], 's-', color='#2ecc71')
        axes[1].set_title('Val Score over Iterations')
        axes[1].set_xlabel('Iteration')
        axes[1].set_ylabel('Val Score (higher = better)')
        plt.tight_layout()
        plt.savefig('results/progress.png', dpi=150)
        plt.show()
        print('\n📊 Chart saved: results/progress.png')
    except ImportError:
        pass
else:
    print('No results.tsv yet — run the training step first.')

## Bonus: Trigger via API (no manual Kaggle session needed)

Run this locally to push a new job to Kaggle automatically:

In [ ]:
# Run locally (not in Kaggle):
# python api-triggers/trigger_kaggle.py \
#   --topic 'Claude skills optimization' \
#   --iterations 20 \
#   --github-repo yourusername/ultimate-doc-researcher \
#   --download-results

# Or trigger via GitHub Actions:
# gh workflow run research.yml \
#   -f topic='Claude skills optimization' \
#   -f iterations=20

print('See AGENTS.md for full trigger instructions')